<a href="https://colab.research.google.com/github/aycakrk/DI725_Ayca/blob/main/Final_Project/phase2_post.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!unzip "/content/drive/MyDrive/RISCM.zip" -d "/content"

Streaming output truncated to the last 5000 lines.
  inflating: /content/resized/RSICD_4364.jpg  
  inflating: /content/resized/NWPU_18440.jpg  
  inflating: /content/resized/NWPU_129.jpg  
  inflating: /content/resized/NWPU_19322.jpg  
  inflating: /content/resized/RSICD_3101.jpg  
  inflating: /content/resized/UCM_2077.jpg  
  inflating: /content/resized/NWPU_10992.jpg  
  inflating: /content/resized/NWPU_10941.jpg  
  inflating: /content/resized/NWPU_19048.jpg  
  inflating: /content/resized/UCM_1535.jpg  
  inflating: /content/resized/NWPU_22896.jpg  
  inflating: /content/resized/NWPU_15332.jpg  
  inflating: /content/resized/RSICD_7475.jpg  
  inflating: /content/resized/NWPU_24431.jpg  
  inflating: /content/resized/NWPU_25180.jpg  
  inflating: /content/resized/RSICD_1536.jpg  
  inflating: /content/resized/NWPU_18103.jpg  
  inflating: /content/resized/NWPU_906.jpg  
  inflating: /content/resized/NWPU_11526.jpg  
  inflating: /content/resized/NWPU_22481.jpg  
  inflating: /con

In [3]:
# Örnek: captions.csv'yi oku
import pandas as pd
df = pd.read_csv('captions.csv')
df.head()


,source,split,image,caption_1,caption_2,caption_3,caption_4,caption_5
0,NWPU,test,NWPU_31430.jpg,A gray plane on the runway and the lawn beside .,A grey plane is on the runway by the lawn .,There is an airplane on the runway with a larg...,A plane is parked on the runway next to the gr...,There is a plane on the runway beside the grass .
1,NWPU,test,NWPU_31431.jpg,Three small planes parked in a line on the air...,"There are four aircraft on the open ground, Th...",There are many planes of different sizes in a ...,Four planes are parked on the runway .,Four planes of different sizes were on the mar...
2,NWPU,test,NWPU_31432.jpg,A plane parked in a line on the airport with s...,A white plane was parked on the instruction li...,An airplane parked in an open area with many c...,A plane is parked on the open space .,There is 1 plane on the ground marked .
3,NWPU,test,NWPU_31433.jpg,A small plane and a big plane parked next to b...,A white plane and a gray plane parked at the b...,Two planes of different sizes are neatly parke...,A large plane and a small plane are parked nea...,Two planes are on the marked ground .
4,NWPU,test,NWPU_31434.jpg,Two planes parked next to boarding bridges .,Two aircraft were parked at the departure gates .,Two planes of different sizes are neatly parke...,Two planes are parked next to the terminal .,Two planes are on the marked ground .


In [4]:
!pip install huggingface_hub
!pip install evaluate transformers tqdm wandb
!pip install rouge_score
!pip install pycocoevalcap


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=503fe7aaf0d385a1fdab6696d46a641e15924c0a96472cc4bd206b93327e5b43
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 24.3 MB/s eta 0:00:00


In [5]:
from huggingface_hub import notebook_login
notebook_login()

In [6]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

# length evaluation
n_train = len(train_df)
n_val   = len(val_df)
n_test  = len(test_df)
total   = n_train + n_val + n_test

print(f"Train: {n_train} ({n_train/total:.2%})")
print(f"Test:  {n_test}  ({n_test/total:.2%})")
print(f"Val:   {n_val}   ({n_val/total:.2%})")

img_dir = '/content/resized'
all_images = set(os.listdir(img_dir))
df_images  = set(df['image'].unique())

missing = sorted(list(df_images - all_images))
if missing:
    print(f"Eksik {len(missing)} dosya var. Örnekler:", missing[:10])
else:
    print("Tüm caption’daki image isimleri klasörde bulundu.")


Train: 35614 (79.99%)
Test:  4454  (10.00%)
Val:   4453   (10.00%)
Tüm caption’daki image isimleri klasörde bulundu.


In [ ]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(output_dir="dummy_dir")

gen_params = [
    "generation_num_beams",
    "generation_no_repeat_ngram_size",
    "generation_repetition_penalty",
    "generation_early_stopping",
    "generation_do_sample",
    "generation_top_p",
    "generation_temperature"
]

for param in gen_params:
    print(f"{param}: Exists = {hasattr(args, param)}, Default = {getattr(args, param, None)}")


generation_num_beams: Exists = True, Default = None
generation_no_repeat_ngram_size: Exists = False, Default = None
generation_repetition_penalty: Exists = False, Default = None
generation_early_stopping: Exists = False, Default = None
generation_do_sample: Exists = False, Default = None
generation_top_p: Exists = False, Default = None
generation_temperature: Exists = False, Default = None


sorun diğer beamleri beklemesi mi?

In [ ]:
import os
import random
from PIL import Image
import wandb

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
    BitsAndBytesConfig,
    GenerationConfig
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from pycocoevalcap.cider.cider import Cider
import pandas as pd

# -------------------------------------------
# 0) WANDB oturumunu başlat (opsiyonel)
# -------------------------------------------
wandb.init(project="DI725_FinalProject", name="lora_r8_eos_gen")

# -------------------------------------------
# 1) Processor & Base Model (8-bit + CPU offload)
# -------------------------------------------
model_name = "google/paligemma-3b-pt-224"
processor  = AutoProcessor.from_pretrained(model_name, use_auth_token=True)
tokenizer  = processor.tokenizer #eklendi

base_model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    device_map="auto",
    use_auth_token=True
)

# Bellek rahatlasın diye gradient checkpointing
base_model.gradient_checkpointing_enable()
base_model.to("cuda")
base_model.eval()

# -------------------------------------------
# 2) Dinamik Caption Dataset
# -------------------------------------------
class RiscDynamicCaptionDataset(Dataset):
    def __init__(self, df, img_dir, processor, max_len=64):
        self.df      = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.proc    = processor
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image"])).convert("RGB")
        caps = [row[f"caption_{i}"] for i in range(1,6)]
        cap  = "<image> " + random.choice(caps) + tokenizer.eos_token  # EOS token eklendi

        enc = self.proc(
            images=img,
            text=cap,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        pixel_values   = enc.pixel_values.squeeze(0)
        input_ids      = enc.input_ids.squeeze(0)
        attention_mask = enc.attention_mask.squeeze(0)

        return {
            "pixel_values":   pixel_values,
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         input_ids.clone()
        }

# -------------------------------------------
# 3) Train / Val Dataset’leri yükle
# -------------------------------------------
img_dir  = "/content/resized"
train_ds = RiscDynamicCaptionDataset(train_df, img_dir, processor)
val_ds   = RiscDynamicCaptionDataset(val_df,   img_dir, processor)

# -------------------------------------------
# 4) LoRA Konfigürasyonu & PEFT Model
# -------------------------------------------
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)
model_lora = get_peft_model(base_model, lora_cfg)
model_lora.print_trainable_parameters()

# -------------------------------------------
# 5) TrainingArguments & Trainer
# -------------------------------------------

training_args = Seq2SeqTrainingArguments(
    output_dir="lora_r8",
    per_device_train_batch_size=3,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=5,
    eval_strategy="epoch",
    # her `logging_steps` adımında log’la
    logging_strategy="steps",
    predict_with_generate=True,
    fp16=True,
    push_to_hub=False,
    report_to="wandb",
    generation_num_beams=1

)


trainer = Seq2SeqTrainer(
    model=model_lora,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    tokenizer=processor
)

# -------------------------------------------
# 6) Eğitim & Değerlendirme
# -------------------------------------------
trainer.train()
lora_metrics = trainer.evaluate()
print("LoRA metrics:", lora_metrics)

# -------------------------------------------
# 7) CIDEr’i ayrı hesaplamak istersen
# -------------------------------------------
preds = trainer.predict(val_ds).predictions
decoded_preds = processor.tokenizer.batch_decode(preds, skip_special_tokens=True)
references = val_df[[f"caption_{i}" for i in range(1,6)]].values.tolist()

refs_dict  = {i: references[i] for i in range(len(references))}
preds_dict = {i: [decoded_preds[i]]     for i in range(len(decoded_preds))}
cider_scorer = Cider()
cider_score, _ = cider_scorer.compute_score(refs_dict, preds_dict)
print("LoRA CIDEr:", cider_score)


# Changed: Pass keyword arguments 'references' and 'predictions' to compute()
bleu_res   = evaluate.load("bleu"  ).compute(references=references, predictions=decoded_preds)["bleu"]
meteor_res = evaluate.load("meteor").compute(references=references, predictions=decoded_preds)["meteor"]
rouge_res  = evaluate.load("rouge" ).compute(references=references, predictions=decoded_preds)["rougeL"]


print(f"BLEU:    {bleu_res:.4f}")
print(f"METEOR:  {meteor_res:.4f}")
print(f"ROUGE-L: {rouge_res:.4f}")
print(f"CIDEr:   {cider_score:.4f}")

wandb.log({
    "bleu":   bleu_res,
    "meteor": meteor_res,
    "rougeL": rouge_res,
    "cider":  cider_score
})


# -------------------------------------------
# 8) Sonuçları CSV’e ve wandbye yaz
# -------------------------------------------
rows = []
for i, img_name in enumerate(val_ds.df["image"].tolist()):
    row = {"image": img_name, "pred": decoded_preds[i]}
    for j, r in enumerate(references[i], start=1):
        row[f"ref_{j}"] = r
    rows.append(row)

pd.DataFrame(rows).to_csv("lora_r8_eos_gen_val_results.csv", index=False)
print("Saved CSV → lora_r8_val_eos_gen_results.csv")

n_samples = 5
# Veri seti uzunluğu
N = len(decoded_preds)
# Rastgele indeksler
sample_idxs = random.sample(range(N), n_samples)

# Bir tablo oluştur
table = wandb.Table(columns=["image", "prediction", "references"])

for i in sample_idxs:
    img_name   = val_ds.df["image"].iloc[i]
    pred       = decoded_preds[i]
    refs       = references[i]  # list of ground-truth’ler
    # referansları tek stringde birleştir
    refs_str   = " || ".join(refs)
    table.add_data(img_name, pred, refs_str)

# Tabloyu logla
wandb.log({"examples": table})





wandb: Currently logged in as: ayca-krk (ayca-krk-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/processing_auto.py:243: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

<ipython-input-4-401bb4857255>:127: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 1,916,928 || all params: 2,925,383,408 || trainable%: 0.0655


...

Epoch,Training Loss,Validation Loss
1,49.547700,12.388762


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_p

LoRA metrics: {'eval_loss': 12.382898330688477, 'eval_runtime': 97.3755, 'eval_samples_per_second': 45.73, 'eval_steps_per_second': 5.72, 'epoch': 1.0}
LoRA CIDEr: 0.24571306323919537


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


BLEU:    0.3825
METEOR:  0.8664
ROUGE-L: 0.8629
CIDEr:   0.2457
Saved CSV → lora_r8_val_eos_gen_results.csv


beam_num=1 yaptım ki tekrarlama sorunu diğr beamleri bekleyene kadar kendince eos tokenı gelmiş olsa bile ekleme yapması mı anlamaktı. ancak 1 desek bile tekrar problemi devam etti.

lets try config again

In [ ]:
import os
import random
from PIL import Image
import wandb

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
    BitsAndBytesConfig,
    GenerationConfig
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from pycocoevalcap.cider.cider import Cider
import pandas as pd

# -------------------------------------------
# 0) WANDB oturumunu başlat (opsiyonel)
# -------------------------------------------
wandb.init(project="DI725_FinalProject", name="lora_r8_eos_gen")

# -------------------------------------------
# 1) Processor & Base Model (8-bit + CPU offload)
# -------------------------------------------
model_name = "google/paligemma-3b-pt-224"
processor  = AutoProcessor.from_pretrained(model_name, use_auth_token=True)
tokenizer  = processor.tokenizer #eklendi

base_model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    device_map="auto",
    use_auth_token=True
)

# Bellek rahatlasın diye gradient checkpointing
base_model.gradient_checkpointing_enable()
base_model.to("cuda")
base_model.eval()

# -------------------------------------------
# 2) Dinamik Caption Dataset
# -------------------------------------------
class RiscDynamicCaptionDataset(Dataset):
    def __init__(self, df, img_dir, processor, max_len=64):
        self.df      = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.proc    = processor
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image"])).convert("RGB")
        caps = [row[f"caption_{i}"] for i in range(1,6)]
        cap  = "<image> " + random.choice(caps) + tokenizer.eos_token  # EOS token eklendi

        enc = self.proc(
            images=img,
            text=cap,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        pixel_values   = enc.pixel_values.squeeze(0)
        input_ids      = enc.input_ids.squeeze(0)
        attention_mask = enc.attention_mask.squeeze(0)

        return {
            "pixel_values":   pixel_values,
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         input_ids.clone()
        }

# -------------------------------------------
# 3) Train / Val Dataset’leri yükle
# -------------------------------------------
img_dir  = "/content/resized"
train_ds = RiscDynamicCaptionDataset(train_df, img_dir, processor)
val_ds   = RiscDynamicCaptionDataset(val_df,   img_dir, processor)

# -------------------------------------------
# 4) LoRA Konfigürasyonu & PEFT Model
# -------------------------------------------
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)
model_lora = get_peft_model(base_model, lora_cfg)
model_lora.print_trainable_parameters()

# -------------------------------------------
# 5) TrainingArguments & Trainer
# -------------------------------------------
gen_cfg = GenerationConfig(
    repetition_penalty=1.5,           # tekrarlanan token’lara 1.5× ceza
    eos_token_id=tokenizer.eos_token_id,
)

training_args = Seq2SeqTrainingArguments(
    output_dir="lora_r8",
    per_device_train_batch_size=3,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=5,
    eval_strategy="epoch",
    # her `logging_steps` adımında log’la
    logging_strategy="steps",
    predict_with_generate=True,
    generation_config=gen_cfg,
    fp16=True,
    push_to_hub=False,
    report_to="wandb"

)


trainer = Seq2SeqTrainer(
    model=model_lora,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    tokenizer=processor
)

# -------------------------------------------
# 6) Eğitim & Değerlendirme
# -------------------------------------------
trainer.train()
lora_metrics = trainer.evaluate()
print("LoRA metrics:", lora_metrics)

# -------------------------------------------
# 7) CIDEr’i ayrı hesaplamak istersen
# -------------------------------------------
preds = trainer.predict(val_ds).predictions
decoded_preds = processor.tokenizer.batch_decode(preds, skip_special_tokens=True)
references = val_df[[f"caption_{i}" for i in range(1,6)]].values.tolist()

refs_dict  = {i: references[i] for i in range(len(references))}
preds_dict = {i: [decoded_preds[i]]     for i in range(len(decoded_preds))}
cider_scorer = Cider()
cider_score, _ = cider_scorer.compute_score(refs_dict, preds_dict)
print("LoRA CIDEr:", cider_score)


# Changed: Pass keyword arguments 'references' and 'predictions' to compute()
bleu_res   = evaluate.load("bleu"  ).compute(references=references, predictions=decoded_preds)["bleu"]
meteor_res = evaluate.load("meteor").compute(references=references, predictions=decoded_preds)["meteor"]
rouge_res  = evaluate.load("rouge" ).compute(references=references, predictions=decoded_preds)["rougeL"]


print(f"BLEU:    {bleu_res:.4f}")
print(f"METEOR:  {meteor_res:.4f}")
print(f"ROUGE-L: {rouge_res:.4f}")
print(f"CIDEr:   {cider_score:.4f}")

wandb.log({
    "bleu":   bleu_res,
    "meteor": meteor_res,
    "rougeL": rouge_res,
    "cider":  cider_score
})


# -------------------------------------------
# 8) Sonuçları CSV’e ve wandbye yaz
# -------------------------------------------
rows = []
for i, img_name in enumerate(val_ds.df["image"].tolist()):
    row = {"image": img_name, "pred": decoded_preds[i]}
    for j, r in enumerate(references[i], start=1):
        row[f"ref_{j}"] = r
    rows.append(row)

pd.DataFrame(rows).to_csv("lora_r8_eos_gen_val_results.csv", index=False)
print("Saved CSV → lora_r8_val_eos_gen_results.csv")

n_samples = 5
# Veri seti uzunluğu
N = len(decoded_preds)
# Rastgele indeksler
sample_idxs = random.sample(range(N), n_samples)

# Bir tablo oluştur
table = wandb.Table(columns=["image", "prediction", "references"])

for i in sample_idxs:
    img_name   = val_ds.df["image"].iloc[i]
    pred       = decoded_preds[i]
    refs       = references[i]  # list of ground-truth’ler
    # referansları tek stringde birleştir
    refs_str   = " || ".join(refs)
    table.add_data(img_name, pred, refs_str)

# Tabloyu logla
wandb.log({"examples": table})





wandb: Currently logged in as: ayca-krk (ayca-krk-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/processing_auto.py:243: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

<ipython-input-4-b9201acbab2a>:131: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 1,916,928 || all params: 2,925,383,408 || trainable%: 0.0655


...

Epoch,Training Loss,Validation Loss
1,49.546000,12.387689


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_p

LoRA metrics: {'eval_loss': 12.381783485412598, 'eval_runtime': 98.0121, 'eval_samples_per_second': 45.433, 'eval_steps_per_second': 5.683, 'epoch': 1.0}
LoRA CIDEr: 1.5370039261867814


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


BLEU:    0.6178
METEOR:  0.9475
ROUGE-L: 0.6449
CIDEr:   1.5370
Saved CSV → lora_r8_val_eos_gen_results.csv


In [ ]:
import os
import random
from PIL import Image
import wandb

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
    BitsAndBytesConfig,
    GenerationConfig
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from pycocoevalcap.cider.cider import Cider
import pandas as pd

# -------------------------------------------
# 0) WANDB oturumunu başlat (opsiyonel)
# -------------------------------------------
wandb.init(project="DI725_FinalProject", name="lora_r8_eos_gen")

# -------------------------------------------
# 1) Processor & Base Model (8-bit + CPU offload)
# -------------------------------------------
model_name = "google/paligemma-3b-pt-224"
processor  = AutoProcessor.from_pretrained(model_name, use_auth_token=True)
tokenizer  = processor.tokenizer #eklendi

base_model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    device_map="auto",
    use_auth_token=True
)

# Bellek rahatlasın diye gradient checkpointing
base_model.gradient_checkpointing_enable()
base_model.to("cuda")
base_model.eval()

# -------------------------------------------
# 2) Dinamik Caption Dataset
# -------------------------------------------
class RiscDynamicCaptionDataset(Dataset):
    def __init__(self, df, img_dir, processor, max_len=64):
        self.df      = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.proc    = processor
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image"])).convert("RGB")
        caps = [row[f"caption_{i}"] for i in range(1,6)]
        cap  = "<image> " + random.choice(caps) + tokenizer.eos_token  # EOS token eklendi

        enc = self.proc(
            images=img,
            text=cap,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        pixel_values   = enc.pixel_values.squeeze(0)
        input_ids      = enc.input_ids.squeeze(0)
        attention_mask = enc.attention_mask.squeeze(0)

        return {
            "pixel_values":   pixel_values,
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         input_ids.clone()
        }

# -------------------------------------------
# 3) Train / Val Dataset’leri yükle
# -------------------------------------------
img_dir  = "/content/resized"
train_ds = RiscDynamicCaptionDataset(train_df, img_dir, processor)
val_ds   = RiscDynamicCaptionDataset(val_df,   img_dir, processor)

# -------------------------------------------
# 4) LoRA Konfigürasyonu & PEFT Model
# -------------------------------------------
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)
model_lora = get_peft_model(base_model, lora_cfg)
model_lora.print_trainable_parameters()

# -------------------------------------------
# 5) TrainingArguments & Trainer
# -------------------------------------------
gen_cfg = GenerationConfig(
    repetition_penalty=1.5,           # tekrarlanan token’lara 1.5× ceza
    num_beams=1,                   # greedy
    no_repeat_ngram_size=2,        # hiçbir 2-gram tekrarına izin verme
    eos_token_id=tokenizer.eos_token_id,
)

training_args = Seq2SeqTrainingArguments(
    output_dir="lora_r8",
    per_device_train_batch_size=3,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=5,
    eval_strategy="epoch",
    # her `logging_steps` adımında log’la
    logging_strategy="steps",
    predict_with_generate=True,
    generation_config=gen_cfg,
    fp16=True,
    push_to_hub=False,
    report_to="wandb",
    max_steps=1500

)


trainer = Seq2SeqTrainer(
    model=model_lora,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    tokenizer=processor
)

# -------------------------------------------
# 6) Eğitim & Değerlendirme
# -------------------------------------------
trainer.train()
lora_metrics = trainer.evaluate()
print("LoRA metrics:", lora_metrics)

# -------------------------------------------
# 7) CIDEr’i ayrı hesaplamak istersen
# -------------------------------------------
preds = trainer.predict(val_ds).predictions
decoded_preds = processor.tokenizer.batch_decode(preds, skip_special_tokens=True)
references = val_df[[f"caption_{i}" for i in range(1,6)]].values.tolist()

refs_dict  = {i: references[i] for i in range(len(references))}
preds_dict = {i: [decoded_preds[i]]     for i in range(len(decoded_preds))}
cider_scorer = Cider()
cider_score, _ = cider_scorer.compute_score(refs_dict, preds_dict)
print("LoRA CIDEr:", cider_score)


# Changed: Pass keyword arguments 'references' and 'predictions' to compute()
bleu_res   = evaluate.load("bleu"  ).compute(references=references, predictions=decoded_preds)["bleu"]
meteor_res = evaluate.load("meteor").compute(references=references, predictions=decoded_preds)["meteor"]
rouge_res  = evaluate.load("rouge" ).compute(references=references, predictions=decoded_preds)["rougeL"]


print(f"BLEU:    {bleu_res:.4f}")
print(f"METEOR:  {meteor_res:.4f}")
print(f"ROUGE-L: {rouge_res:.4f}")
print(f"CIDEr:   {cider_score:.4f}")

wandb.log({
    "bleu":   bleu_res,
    "meteor": meteor_res,
    "rougeL": rouge_res,
    "cider":  cider_score
})


# -------------------------------------------
# 8) Sonuçları CSV’e ve wandbye yaz
# -------------------------------------------
rows = []
for i, img_name in enumerate(val_ds.df["image"].tolist()):
    row = {"image": img_name, "pred": decoded_preds[i]}
    for j, r in enumerate(references[i], start=1):
        row[f"ref_{j}"] = r
    rows.append(row)

pd.DataFrame(rows).to_csv("lora_r8_eos_gen_val_results.csv", index=False)
print("Saved CSV → lora_r8_val_eos_gen_results.csv")

n_samples = 5
# Veri seti uzunluğu
N = len(decoded_preds)
# Rastgele indeksler
sample_idxs = random.sample(range(N), n_samples)

# Bir tablo oluştur
table = wandb.Table(columns=["image", "prediction", "references"])

for i in sample_idxs:
    img_name   = val_ds.df["image"].iloc[i]
    pred       = decoded_preds[i]
    refs       = references[i]  # list of ground-truth’ler
    # referansları tek stringde birleştir
    refs_str   = " || ".join(refs)
    table.add_data(img_name, pred, refs_str)

# Tabloyu logla
wandb.log({"examples": table})





wandb: Currently logged in as: ayca-krk (ayca-krk-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/processing_auto.py:243: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

<ipython-input-5-3515243c4420>:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 1,916,928 || all params: 2,925,383,408 || trainable%: 0.0655


...

Epoch,Training Loss,Validation Loss
0,49.765200,12.433043


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(


LoRA metrics: {'eval_loss': 12.43194580078125, 'eval_runtime': 97.6634, 'eval_samples_per_second': 45.595, 'eval_steps_per_second': 5.703, 'epoch': 0.5053908355795148}
LoRA CIDEr: 0.8699742532201853


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


BLEU:    0.4783
METEOR:  0.9147
ROUGE-L: 0.8890
CIDEr:   0.8700
Saved CSV → lora_r8_val_eos_gen_results.csv


genlerden anlamıyo, acaba eğitim noktasında müdahele etmek daha mı iyi olur? pad_token:

In [ ]:
import os
import random
from PIL import Image
import wandb

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
    BitsAndBytesConfig,
    GenerationConfig
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from pycocoevalcap.cider.cider import Cider
import pandas as pd
from transformers import DataCollatorForSeq2Seq


# -------------------------------------------
# 0) WANDB oturumunu başlat (opsiyonel)
# -------------------------------------------
wandb.init(project="DI725_FinalProject", name="lora_r8_eos_pad")

# -------------------------------------------
# 1) Processor & Base Model (8-bit + CPU offload)
# -------------------------------------------
model_name = "google/paligemma-3b-pt-224"
processor  = AutoProcessor.from_pretrained(model_name, use_auth_token=True)
tokenizer  = processor.tokenizer #eklendi

base_model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    device_map="auto",
    use_auth_token=True
)

# Bellek rahatlasın diye gradient checkpointing
base_model.gradient_checkpointing_enable()
base_model.to("cuda")
base_model.eval()

# -------------------------------------------
# 2) Dinamik Caption Dataset
# -------------------------------------------
class RiscDynamicCaptionDataset(Dataset):
    def __init__(self, df, img_dir, processor, max_len=64):
        self.df      = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.proc    = processor
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image"])).convert("RGB")
        caps = [row[f"caption_{i}"] for i in range(1,6)]
        cap  = "<image> " + random.choice(caps) + tokenizer.eos_token  # EOS token eklendi

        enc = self.proc(
            images=img,
            text=cap,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        pixel_values   = enc.pixel_values.squeeze(0)
        input_ids      = enc.input_ids.squeeze(0)
        attention_mask = enc.attention_mask.squeeze(0)

        return {
            "pixel_values":   pixel_values,
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         input_ids.clone()
        }

# -------------------------------------------
# 3) Train / Val Dataset’leri yükle
# -------------------------------------------
img_dir  = "/content/resized"
train_ds = RiscDynamicCaptionDataset(train_df, img_dir, processor)
val_ds   = RiscDynamicCaptionDataset(val_df,   img_dir, processor)

# -------------------------------------------
# 4) LoRA Konfigürasyonu & PEFT Model
# -------------------------------------------
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)
model_lora = get_peft_model(base_model, lora_cfg)
model_lora.print_trainable_parameters()

# -------------------------------------------
# 5) TrainingArguments & Trainer
# -------------------------------------------

training_args = Seq2SeqTrainingArguments(
    output_dir="lora_r8",
    per_device_train_batch_size=3,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=5,
    eval_strategy="epoch",
    # her `logging_steps` adımında log’la
    logging_strategy="steps",
    predict_with_generate=True,
    fp16=True,
    push_to_hub=False,
    report_to="wandb",
    max_steps=1500

)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=processor.tokenizer,
    label_pad_token_id=-100
)

trainer = Seq2SeqTrainer(
    model=model_lora,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    tokenizer=processor
)

# -------------------------------------------
# 6) Eğitim & Değerlendirme
# -------------------------------------------
trainer.train()
lora_metrics = trainer.evaluate()
print("LoRA metrics:", lora_metrics)

# -------------------------------------------
# 7) CIDEr’i ayrı hesaplamak istersen
# -------------------------------------------
preds = trainer.predict(val_ds).predictions
decoded_preds = processor.tokenizer.batch_decode(preds, skip_special_tokens=True)
references = val_df[[f"caption_{i}" for i in range(1,6)]].values.tolist()

refs_dict  = {i: references[i] for i in range(len(references))}
preds_dict = {i: [decoded_preds[i]]     for i in range(len(decoded_preds))}
cider_scorer = Cider()
cider_score, _ = cider_scorer.compute_score(refs_dict, preds_dict)
print("LoRA CIDEr:", cider_score)


# Changed: Pass keyword arguments 'references' and 'predictions' to compute()
bleu_res   = evaluate.load("bleu"  ).compute(references=references, predictions=decoded_preds)["bleu"]
meteor_res = evaluate.load("meteor").compute(references=references, predictions=decoded_preds)["meteor"]
rouge_res  = evaluate.load("rouge" ).compute(references=references, predictions=decoded_preds)["rougeL"]


print(f"BLEU:    {bleu_res:.4f}")
print(f"METEOR:  {meteor_res:.4f}")
print(f"ROUGE-L: {rouge_res:.4f}")
print(f"CIDEr:   {cider_score:.4f}")

wandb.log({
    "bleu":   bleu_res,
    "meteor": meteor_res,
    "rougeL": rouge_res,
    "cider":  cider_score
})


# -------------------------------------------
# 8) Sonuçları CSV’e ve wandbye yaz
# -------------------------------------------
rows = []
for i, img_name in enumerate(val_ds.df["image"].tolist()):
    row = {"image": img_name, "pred": decoded_preds[i]}
    for j, r in enumerate(references[i], start=1):
        row[f"ref_{j}"] = r
    rows.append(row)

pd.DataFrame(rows).to_csv("lora_r8_eos_pad_val_results.csv", index=False)
print("Saved CSV → lora_r8_val_eos_pad_results.csv")

n_samples = 5
# Veri seti uzunluğu
N = len(decoded_preds)
# Rastgele indeksler
sample_idxs = random.sample(range(N), n_samples)

# Bir tablo oluştur
table = wandb.Table(columns=["image", "prediction", "references"])

for i in sample_idxs:
    img_name   = val_ds.df["image"].iloc[i]
    pred       = decoded_preds[i]
    refs       = references[i]  # list of ground-truth’ler
    # referansları tek stringde birleştir
    refs_str   = " || ".join(refs)
    table.add_data(img_name, pred, refs_str)

# Tabloyu logla
wandb.log({"examples": table})





wandb: Currently logged in as: ayca-krk (ayca-krk-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/processing_auto.py:243: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

<ipython-input-4-ced1de550bd9>:133: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 1,916,928 || all params: 2,925,383,408 || trainable%: 0.0655


...

Epoch,Training Loss,Validation Loss
0,49.784300,12.437641


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(


LoRA metrics: {'eval_loss': 12.436532020568848, 'eval_runtime': 226.7275, 'eval_samples_per_second': 19.64, 'eval_steps_per_second': 2.457, 'epoch': 0.5053908355795148}
LoRA CIDEr: 0.81478205666173


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


BLEU:    0.4599
METEOR:  0.8892
ROUGE-L: 0.7479
CIDEr:   0.8148
Saved CSV → lora_r8_val_eos_pad_results.csv


this didnt work either

It seems that this system does not understand from the inputs given to the config section that it should not repeat words, and this will not solve the word repeat problem. In this case, we will solve the problem not in the generation section, but in post-processing.

In [7]:
import os
import random
from PIL import Image
import wandb

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
    BitsAndBytesConfig,
    GenerationConfig
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from pycocoevalcap.cider.cider import Cider
import pandas as pd

# -------------------------------------------
# 0) WANDB oturumunu başlat (opsiyonel)
# -------------------------------------------
wandb.init(project="DI725_FinalProject", name="lora_r8_eos_post")

# -------------------------------------------
# 1) Processor & Base Model (8-bit + CPU offload)
# -------------------------------------------
model_name = "google/paligemma-3b-pt-224"
processor  = AutoProcessor.from_pretrained(model_name, use_auth_token=True)
tokenizer  = processor.tokenizer #eklendi

base_model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    device_map="auto",
    use_auth_token=True
)

# Bellek rahatlasın diye gradient checkpointing
base_model.gradient_checkpointing_enable()
base_model.to("cuda")
base_model.eval()

# -------------------------------------------
# 2) Dinamik Caption Dataset
# -------------------------------------------
class RiscDynamicCaptionDataset(Dataset):
    def __init__(self, df, img_dir, processor, max_len=64):
        self.df      = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.proc    = processor
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image"])).convert("RGB")
        caps = [row[f"caption_{i}"] for i in range(1,6)]
        cap  = "<image> " + random.choice(caps) + tokenizer.eos_token  # EOS token eklendi

        enc = self.proc(
            images=img,
            text=cap,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        pixel_values   = enc.pixel_values.squeeze(0)
        input_ids      = enc.input_ids.squeeze(0)
        attention_mask = enc.attention_mask.squeeze(0)

        return {
            "pixel_values":   pixel_values,
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         input_ids.clone()
        }

# -------------------------------------------
# 3) Train / Val Dataset’leri yükle
# -------------------------------------------
img_dir  = "/content/resized"
train_ds = RiscDynamicCaptionDataset(train_df, img_dir, processor)
val_ds   = RiscDynamicCaptionDataset(val_df,   img_dir, processor)

# -------------------------------------------
# 4) LoRA Konfigürasyonu & PEFT Model
# -------------------------------------------
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)
model_lora = get_peft_model(base_model, lora_cfg)
model_lora.print_trainable_parameters()

# -------------------------------------------
# 5) TrainingArguments & Trainer
# -------------------------------------------

training_args = Seq2SeqTrainingArguments(
    output_dir="lora_r8",
    per_device_train_batch_size=3,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=5,
    eval_strategy="epoch",
    # her `logging_steps` adımında log’la
    logging_strategy="steps",
    predict_with_generate=True,
    fp16=True,
    push_to_hub=False,
    report_to="wandb",
    max_steps=1500

)


trainer = Seq2SeqTrainer(
    model=model_lora,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    tokenizer=processor
)

# -------------------------------------------
# 6) Eğitim & Değerlendirme
# -------------------------------------------
trainer.train()
lora_metrics = trainer.evaluate()
print("LoRA metrics:", lora_metrics)

raw_pred_ids = trainer.predict(val_ds).predictions  # shape: (N, seq_len)

clean_preds = []
for seq_ids in raw_pred_ids:
    # 1) İlk EOS token’ın index’ini bul
    #    (eğer eos_token_id hiç yoksa bütün seq’i kullan)
    ids = list(seq_ids)
    if tokenizer.eos_token_id in ids:
        cut = ids.index(tokenizer.eos_token_id)
        ids = ids[:cut]
    # 2) Decode & special tokens’ı at
    text = tokenizer.decode(ids, skip_special_tokens=True).strip()
    clean_preds.append(text)

# Artık clean_preds, "<pad>" ve "<image>" içermiyor,
# EOS sonrası dolguları da kesinlikle bırakmıyor.

# references
references = val_df[[f"caption_{i}" for i in range(1,6)]].values.tolist()

# CIDEr
refs_dict  = {i: references[i] for i in range(len(references))}
preds_dict = {i: [clean_preds[i]]       for i in range(len(clean_preds))}
cider_scorer = Cider()
cider_score, _ = cider_scorer.compute_score(refs_dict, preds_dict)
print("LoRA CIDEr:", cider_score)

# BLEU / METEOR / ROUGE
bleu_res   = evaluate.load("bleu" ).compute(references=references, predictions=clean_preds)["bleu"]
meteor_res = evaluate.load("meteor").compute(references=references, predictions=clean_preds)["meteor"]
rouge_res  = evaluate.load("rouge" ).compute(references=references, predictions=clean_preds)["rougeL"]

print(f"BLEU:    {bleu_res:.4f}")
print(f"METEOR:  {meteor_res:.4f}")
print(f"ROUGE-L: {rouge_res:.4f}")
print(f"CIDEr:   {cider_score:.4f}")

wandb.log({
    "bleu":   bleu_res,
    "meteor": meteor_res,
    "rougeL": rouge_res,
    "cider":  cider_score
})


# CSV ve wandb örnekleri için de clean_preds kullan
rows = []
for i, img_name in enumerate(val_ds.df["image"].tolist()):
    row = {"image": img_name, "pred": clean_preds[i]}
    for j, r in enumerate(references[i], start=1):
        row[f"ref_{j}"] = r
    rows.append(row)
pd.DataFrame(rows).to_csv("lora_r8_eos_post_val_results.csv", index=False)
print("Saved CSV → lora_r8_eos_post_val_results.csv")

table = wandb.Table(columns=["image", "prediction", "references"])
for idx in random.sample(range(len(clean_preds)), 5):
    table.add_data(val_ds.df["image"].iloc[idx],
                   clean_preds[idx],
                   " || ".join(references[idx]))
wandb.log({"examples": table})





<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayca-krk (ayca-krk-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/processing_auto.py:243: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

Epoch,Training Loss,Validation Loss
0,49.759700,12.431544


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/configuration_paligemma.py:135: FutureWarning: The `ignore_index` attribute is deprecated and will be removed in v4.47.
  warnings.warn(


LoRA metrics: {'eval_loss': 12.430432319641113, 'eval_runtime': 97.2718, 'eval_samples_per_second': 45.779, 'eval_steps_per_second': 5.726, 'epoch': 0.5053908355795148}
LoRA CIDEr: 3.1215789050485263


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


BLEU:    1.0000
METEOR:  0.9995
ROUGE-L: 1.0000
CIDEr:   3.1216
Saved CSV → lora_r8_eos_post_val_results.csv


as it can be seen, with the post processing, the model got full score from some metrics. the words created after eos are erased

In [8]:
# 9) Gerçek skorları hesapla
import evaluate
from pycocoevalcap.cider.cider import Cider

bleu_metric   = evaluate.load("bleu")
meteor_metric = evaluate.load("meteor")
rouge_metric  = evaluate.load("rouge")
cider_scorer  = Cider()

bleu   = bleu_metric.compute(predictions=clean_preds, references=references)["bleu"]
meteor = meteor_metric.compute(predictions=clean_preds, references=references)["meteor"]
rouge  = rouge_metric.compute(predictions=clean_preds, references=references)["rougeL"]
cider, _ = cider_scorer.compute_score(
    {i: references[i]       for i in range(len(references))},
    {i: [clean_preds[i]]    for i in range(len(clean_preds))}
)

# 10) “Max possible” skorlar için perfect tahminleri hazırla
def most_common_ref(ref_list):
    sets   = [set(r.split()) for r in ref_list]
    scores = [
        sum(len(sets[i].intersection(sets[j])) for j in range(len(sets)) if j != i)
        for i in range(len(sets))
    ]
    return ref_list[scores.index(max(scores))]

perfect_preds = [most_common_ref(r) for r in references]

max_bleu   = bleu_metric.compute(predictions=perfect_preds, references=references)["bleu"]
max_meteor = meteor_metric.compute(predictions=perfect_preds, references=references)["meteor"]
max_rouge  = rouge_metric.compute(predictions=perfect_preds, references=references)["rougeL"]
max_cider, _ = cider_scorer.compute_score(
    {i: references[i]        for i in range(len(references))},
    {i: [perfect_preds[i]]   for i in range(len(perfect_preds))}
)

# 11) Skorları yazdır
print("              SCORE   /   MAX POSSIBLE")
print(f"BLEU:    {bleu:.4f} / {max_bleu:.4f}")
print(f"METEOR:  {meteor:.4f} / {max_meteor:.4f}")
print(f"ROUGE-L: {rouge:.4f} / {max_rouge:.4f}")
print(f"CIDEr:   {cider:.4f} / {max_cider:.4f}")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


              SCORE   /   MAX POSSIBLE
BLEU:    1.0000 / 1.0000
METEOR:  0.9995 / 0.9997
ROUGE-L: 1.0000 / 1.0000
CIDEr:   3.1216 / 3.3915
